## Part 2 — More Advanced Problems with Solutions

This section continues with new problems on membership optimization, bytecode inspection, benchmarking traps, and optimizer-safe reasoning.

In [1]:
import dis
import sys
import timeit
from types import CodeType

print(sys.version)


3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]


## Problem 3 — Membership-test rewriting

CPython may rewrite some literal containers used only for membership testing.

**Tasks**

1. Compare membership tests against a list, tuple, set, and frozenset literal.
2. Inspect `co_consts`.
3. Explain which containers are stored as constants and why.

In [2]:
def membership_list(x):
    return x in [10, 20, 30]

def membership_tuple(x):
    return x in (10, 20, 30)

def membership_set(x):
    return x in {10, 20, 30}

def membership_frozenset(x):
    return x in frozenset({10, 20, 30})

for fn in [membership_list, membership_tuple, membership_set, membership_frozenset]:
    print(fn.__name__)
    print(fn.__code__.co_consts)
    dis.dis(fn)
    print('-' * 60)


membership_list
(None, (10, 20, 30))
  1           RESUME                   0

  2           LOAD_FAST                0 (x)
              LOAD_CONST               1 ((10, 20, 30))
              CONTAINS_OP              0
              RETURN_VALUE
------------------------------------------------------------
membership_tuple
(None, (10, 20, 30))
  4           RESUME                   0

  5           LOAD_FAST                0 (x)
              LOAD_CONST               1 ((10, 20, 30))
              CONTAINS_OP              0
              RETURN_VALUE
------------------------------------------------------------
membership_set
(None, frozenset({10, 20, 30}))
  7           RESUME                   0

  8           LOAD_FAST                0 (x)
              LOAD_CONST               1 (frozenset({10, 20, 30}))
              CONTAINS_OP              0
              RETURN_VALUE
------------------------------------------------------------
membership_frozenset
(None, frozenset({10, 20, 30})

### Solution 3

A list literal used only in `x in [...]` can safely become a tuple constant because the list itself is never exposed to user code.

A set literal used only in `x in {...}` can safely become a `frozenset` constant. This is usually faster for membership lookup and avoids constructing a new mutable set on every call.

A tuple literal is already immutable, so it can be stored directly as a tuple constant.

`frozenset({10, 20, 30})` is different: this is a function call. The compiler generally cannot replace arbitrary calls with constants, even if the call looks obvious to a human. Calls can be shadowed, monkey-patched, or have effects in less trivial cases.

In [3]:
assert (10, 20, 30) in membership_list.__code__.co_consts
assert (10, 20, 30) in membership_tuple.__code__.co_consts
assert any(type(c) is frozenset and c == frozenset({10, 20, 30}) for c in membership_set.__code__.co_consts)

# The explicit frozenset(...) expression is expected to involve a runtime name lookup/call.
assert 'frozenset' in membership_frozenset.__code__.co_names


## Problem 4 — Name lookup prevents folding

The compiler folds expressions made from constants, but names are not constants.

**Tasks**

1. Compare two functions: one using only literals, one using a named variable.
2. Inspect bytecode.
3. Explain why the optimizer must treat them differently.

In [4]:
SECONDS_PER_MINUTE = 60

def literal_seconds():
    return 24 * 60 * 60

def named_seconds():
    return 24 * 60 * SECONDS_PER_MINUTE

for fn in [literal_seconds, named_seconds]:
    print(fn.__name__)
    print('co_consts:', fn.__code__.co_consts)
    print('co_names:', fn.__code__.co_names)
    dis.dis(fn)
    print('-' * 60)


literal_seconds
co_consts: (None, 86400)
co_names: ()
  3           RESUME                   0

  4           RETURN_CONST             1 (86400)
------------------------------------------------------------
named_seconds
co_consts: (None, 1440)
co_names: ('SECONDS_PER_MINUTE',)
  6           RESUME                   0

  7           LOAD_CONST               1 (1440)
              LOAD_GLOBAL              0 (SECONDS_PER_MINUTE)
              BINARY_OP                5 (*)
              RETURN_VALUE
------------------------------------------------------------


### Solution 4

`literal_seconds` can be folded to a single integer constant because the entire expression is made from literals.

`named_seconds` cannot be fully folded because `SECONDS_PER_MINUTE` is a name lookup. The global name could be rebound before the function is called.

This is why Python optimizes conservatively. The compiler must preserve dynamic language semantics, even when a value appears stable in the source file.

In [5]:
assert 86400 in literal_seconds.__code__.co_consts
assert 'SECONDS_PER_MINUTE' in named_seconds.__code__.co_names

old_value = SECONDS_PER_MINUTE
SECONDS_PER_MINUTE = 100
try:
    assert literal_seconds() == 86400
    assert named_seconds() == 24 * 60 * 100  # 144000
finally:
    SECONDS_PER_MINUTE = old_value

## Problem 5 — Dead-code elimination after constant conditions

Some unreachable branches can disappear during compilation.

**Tasks**

1. Inspect the constants and bytecode below.
2. Determine whether the unreachable code remains.
3. Explain why this is safe for literal conditions but not for arbitrary expressions.

In [6]:
def constant_false_branch():
    if False:
        return 'unreachable'
    return 'reachable'

def constant_true_branch():
    if True:
        return 'reachable'
    return 'unreachable'

for fn in [constant_false_branch, constant_true_branch]:
    print(fn.__name__)
    print(fn.__code__.co_consts)
    dis.dis(fn)
    print('-' * 60)


constant_false_branch
(None, 'reachable')
  1           RESUME                   0

  2           NOP

  4           RETURN_CONST             1 ('reachable')
------------------------------------------------------------
constant_true_branch
(None, 'reachable')
  6           RESUME                   0

  7           NOP

  8           RETURN_CONST             1 ('reachable')
------------------------------------------------------------


### Solution 5

Branches guarded by literal `True` or literal `False` can be simplified because their truth value is known at compile time.

The unreachable branch should not execute, and modern CPython can omit much of it from executable bytecode.

This differs from something like `if some_function(): ...`. Even if a human expects the function to return `False`, the compiler cannot assume that without changing program behavior.

In [7]:
assert constant_false_branch() == 'reachable'
assert constant_true_branch() == 'reachable'

assert 'reachable' in constant_false_branch.__code__.co_consts
assert 'reachable' in constant_true_branch.__code__.co_consts


## Problem 6 — Benchmarking optimized membership correctly

A benchmark can accidentally measure construction cost instead of membership cost.

**Tasks**

1. Compare membership in a literal set inside a function with membership in a prebuilt global set.
2. Explain why the results may be closer than expected.
3. Compare that with a dynamically built set.

In [8]:
PREBUILT = set(range(100))

def literal_set_membership(x):
    return x in {0, 1, 2, 3, 4, 5, 6, 7, 8, 9,
                 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
                 20, 21, 22, 23, 24, 25, 26, 27, 28, 29,
                 30, 31, 32, 33, 34, 35, 36, 37, 38, 39,
                 40, 41, 42, 43, 44, 45, 46, 47, 48, 49,
                 50, 51, 52, 53, 54, 55, 56, 57, 58, 59,
                 60, 61, 62, 63, 64, 65, 66, 67, 68, 69,
                 70, 71, 72, 73, 74, 75, 76, 77, 78, 79,
                 80, 81, 82, 83, 84, 85, 86, 87, 88, 89,
                 90, 91, 92, 93, 94, 95, 96, 97, 98, 99}

def prebuilt_set_membership(x):
    return x in PREBUILT

def dynamic_set_membership(x):
    return x in set(range(100))

for fn in [literal_set_membership, prebuilt_set_membership, dynamic_set_membership]:
    t = timeit.timeit(lambda: fn(99), number=300_000)
    print(f'{fn.__name__:28s} {t:.6f}')


literal_set_membership       0.050194
prebuilt_set_membership      0.053397
dynamic_set_membership       1.069155


### Solution 6

`literal_set_membership` may be optimized to use a `frozenset` constant. That means the benchmark is largely measuring membership lookup, not set construction.

`prebuilt_set_membership` uses a global set object. It has a global name lookup plus a set membership test.

`dynamic_set_membership` constructs a new set from `range(100)` on every call, so it measures much more than membership. This is the benchmark trap: similar-looking code can test very different things.

In [9]:
assert any(type(c) is frozenset for c in literal_set_membership.__code__.co_consts)
assert 'PREBUILT' in prebuilt_set_membership.__code__.co_names
assert 'set' in dynamic_set_membership.__code__.co_names
assert 'range' in dynamic_set_membership.__code__.co_names
